# G01 — Analyse des Résultats du Random Search
**Projet** : Fine-tuning de Transformers  
**Groupe** : G01 | P01 Benchmark d'Optimiseurs | Random Search

⚠️ **Prérequis** : Exécuter `python main.py` avant d'utiliser ce notebook.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
print('Prêt ✓')

## 1. Chargement des résultats

In [ ]:
results = pd.read_csv('../outputs/results/random_search_results.csv')
with open('../outputs/results/best_config.json') as f:
    best = json.load(f)

print(f'Configurations testées : {len(results)}')
print('\nAperçu :')
results.style.highlight_max(subset=['best_val_acc', 'test_accuracy', 'test_f1'],
                             color='#d4edda')

## 2. Tableau comparatif par optimiseur

In [ ]:
summary = results.groupby('optimizer').agg(
    best_val_acc  = ('best_val_acc',   'max'),
    mean_val_acc  = ('best_val_acc',   'mean'),
    best_test_acc = ('test_accuracy',  'max'),
    best_f1       = ('test_f1',        'max'),
    mean_time_s   = ('total_time_s',   'mean'),
).round(4).reset_index()

print('Résumé par optimiseur :')
print(summary.to_string(index=False))

## 3. Meilleure configuration

In [ ]:
print('★ MEILLEURE CONFIGURATION')
print('─' * 40)
for k, v in best.items():
    if isinstance(v, float):
        print(f'  {k:20s} : {v:.4f}')
    else:
        print(f'  {k:20s} : {v}')

## 4. Visualisation : Val Accuracy par optimiseur × LR

In [ ]:
colors = {'AdamW': '#2E75B6', 'SGD': '#E07B39', 'Adafactor': '#43A047'}

fig, ax = plt.subplots(figsize=(10, 5))
for opt, grp in results.groupby('optimizer'):
    ax.scatter(grp['learning_rate'], grp['best_val_acc'],
               label=opt, color=colors.get(opt, '#888'),
               s=120, alpha=0.85, zorder=3)

ax.set_xscale('log')
ax.set_xlabel('Learning Rate (échelle log)')
ax.set_ylabel('Best Val Accuracy')
ax.set_title('Random Search — Val Accuracy × Learning Rate par Optimiseur')
ax.legend(title='Optimiseur')
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Métriques de platitude (Sharpness)

In [ ]:
try:
    landscape = pd.read_csv('../outputs/results/landscape_metrics.csv')
    print('Sharpness par configuration :')
    print(landscape.sort_values('sharpness').to_string(index=False))
    print('\n→ Une sharpness plus basse indique un minimum plus plat,')
    print('  ce qui est associé à une meilleure généralisation (Keskar et al., 2017).')
except FileNotFoundError:
    print('Fichier landscape_metrics.csv non trouvé. Lancez main.py en premier.')

## 6. Conclusions préliminaires

In [ ]:
best_opt = summary.loc[summary['best_val_acc'].idxmax(), 'optimizer']
fastest  = summary.loc[summary['mean_time_s'].idxmin(),  'optimizer']

print(f'✅ Optimiseur le plus performant (val) : {best_opt}')
print(f'⚡ Optimiseur le plus rapide           : {fastest}')
print()
print('Recommandation : Pour le fine-tuning de DistilBERT sur IMDb,')
print(f'  {best_opt} avec un LR autour de {best["learning_rate"]:.2e} semble optimal.')